# BC-only

In [8]:
!pip install optuna

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [9]:
dqn_model_variation = 'BC_only'

In [10]:
import os
import sys
import yaml
import pandas as pd
from optuna.terminator.improvement.emmr import torch

if 'google.colab' in sys.modules:
  from google.colab import drive
  drive.mount( "/content/drive")
  if os.path.isdir(f'drive/MyDrive/Offline_RL_BSc_Thesis_repo/Offline_RL_BSc_Thesis/notebooks/DQN/{dqn_model_variation}'):
    os.chdir(f'drive/MyDrive/Offline_RL_BSc_Thesis_repo/Offline_RL_BSc_Thesis/notebooks/DQN/{dqn_model_variation}')


project_root = os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

with open(f'../../../config/DQN/{dqn_model_variation}/{dqn_model_variation.lower()}_experiments_config.yaml', 'r') as f:
    experiments_config = yaml.safe_load(f)

torch.manual_seed(experiments_config['experiment']['seed'])

selected_features = ['X', 'Y', 'lv_X', 'lv_Y',
                     'angle', 'angular_velocity', 'leg_1', 'leg_2',
                     'reward',
                     'next_X', 'next_Y', 'next_lv_X', 'next_lv_Y',
                     'next_angle', 'next_angular_velocity', 'next_leg_1', 'next_leg_2']

In [11]:
import warnings
warnings.filterwarnings('ignore')

## Hyperparameter Tuning and Training

In [12]:
from src.experiments.DQN_BC_experiments import conduct_dqn_bc_experiment

### Replay Buffer

In [13]:
rb_train_df = pd.read_parquet('../../../data/replay_buffer_episodes/rb_train_next_states_included.parquet').drop(columns=['episode'])
rb_train_state_reward_nex_state_df = rb_train_df.drop(columns=['done'])
rb_train_dones_tensor = torch.tensor(rb_train_df['done']).float()

rb_normalization_techniques = {
    'raw': None,
    'max_abs': torch.jit.load(f'../../../models/DQN/replay_buffer/normalization/max_abs_normalization.pt'),
    'min_max': torch.jit.load(f'../../../models/DQN/replay_buffer/normalization/min_max_normalization.pt'),
    'robust': torch.jit.load(f'../../../models/DQN/replay_buffer/normalization/robust_normalization.pt'),
    'standard': torch.jit.load(f'../../../models/DQN/replay_buffer/normalization/standard_normalization.pt'),
}

rb_generative_models_scripts = {
    'raw': torch.jit.load(f'../../../models/BC/replay_buffer/BC_raw.pt'),
    'max_abs': torch.jit.load(f'../../../models/BC/replay_buffer/BC_max_abs.pt'),
    'min_max': torch.jit.load(f'../../../models/BC/replay_buffer/BC_min_max.pt'),
    'robust': torch.jit.load(f'../../../models/BC/replay_buffer/BC_robust.pt'),
    'standard': torch.jit.load(f'../../../models/BC/replay_buffer/BC_standard.pt'),
}

#### Raw

In [14]:
norm_name = 'raw'

conduct_dqn_bc_experiment(
    dataset_name='replay_buffer',
    norm_technique_name=norm_name,
    norm_technique_script=rb_normalization_techniques[norm_name],
    generative_model_script=rb_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=rb_train_state_reward_nex_state_df,
    dones=rb_train_dones_tensor,
    experiments_config=experiments_config,
)

[I 2026-01-16 14:09:26,439] A new study created in RDB with name: dqn_bc_replay_buffer_raw_data_study


Study 'dqn_bc_replay_buffer_raw_data_study' already has 0 trials. Running 15 more...


Trial 0:  41%|████      | 61000/150000 [04:00<05:51, 253.51it/s, loss=3.4]  
[I 2026-01-16 14:13:28,786] Trial 0 finished with value: 0.11134159564971924 and parameters: {'lr': 0.0007884231006942208, 'num_hidden_neurons': 128, 'num_hidden_layers': 1}. Best is trial 0 with value: 0.11134159564971924.
Trial 1:  35%|███▍      | 52000/150000 [04:29<08:27, 193.02it/s, loss=0.742]
[I 2026-01-16 14:17:58,265] Trial 1 finished with value: 0.1159210279583931 and parameters: {'lr': 0.00019930908144115552, 'num_hidden_neurons': 64, 'num_hidden_layers': 3}. Best is trial 0 with value: 0.11134159564971924.
Trial 2:  57%|█████▋    | 86000/150000 [10:02<07:28, 142.68it/s, loss=1.33]  
[I 2026-01-16 14:28:01,097] Trial 2 finished with value: 0.07900557667016983 and parameters: {'lr': 0.00045873568341913757, 'num_hidden_neurons': 64, 'num_hidden_layers': 6}. Best is trial 2 with value: 0.07900557667016983.
Trial 3:  49%|████▉     | 74000/150000 [06:06<06:16, 201.98it/s, loss=0.493] 
[I 2026-01-16 14:34

#### Max-Abs

In [15]:
norm_name = 'max_abs'

conduct_dqn_bc_experiment(
    dataset_name='replay_buffer',
    norm_technique_name=norm_name,
    norm_technique_script=rb_normalization_techniques[norm_name],
    generative_model_script=rb_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=rb_train_state_reward_nex_state_df,
    dones=rb_train_dones_tensor,
    experiments_config=experiments_config,
)

Trial 13:   4%|▍         | 6000/150000 [00:52<20:56, 114.65it/s, loss=0.409]
[I 2026-01-16 14:43:38,505] A new study created in RDB with name: dqn_bc_replay_buffer_max_abs_data_study


Study 'dqn_bc_replay_buffer_max_abs_data_study' already has 0 trials. Running 15 more...


Trial 0:  51%|█████     | 76000/150000 [08:47<08:33, 144.05it/s, loss=0.000974] 
[I 2026-01-16 14:52:26,208] Trial 0 finished with value: 1.2445266293070745e-05 and parameters: {'lr': 0.0007454820492874843, 'num_hidden_neurons': 128, 'num_hidden_layers': 6}. Best is trial 0 with value: 1.2445266293070745e-05.
Trial 1:  43%|████▎     | 64000/150000 [06:52<09:14, 155.14it/s, loss=0.00279] 
[I 2026-01-16 14:59:18,822] Trial 1 finished with value: 1.499312202213332e-05 and parameters: {'lr': 0.0005439560977950787, 'num_hidden_neurons': 64, 'num_hidden_layers': 6}. Best is trial 0 with value: 1.2445266293070745e-05.
Trial 2:  30%|███       | 45000/150000 [03:22<07:52, 222.09it/s, loss=0.000108] 
[I 2026-01-16 15:02:41,529] Trial 2 finished with value: 1.6375191989936866e-05 and parameters: {'lr': 0.0007912112966695806, 'num_hidden_neurons': 96, 'num_hidden_layers': 2}. Best is trial 0 with value: 1.2445266293070745e-05.
Trial 3:  29%|██▊       | 43000/150000 [03:19<08:16, 215.45it/s, loss=3

#### Min-Max

In [16]:
norm_name = 'min_max'

conduct_dqn_bc_experiment(
    dataset_name='replay_buffer',
    norm_technique_name=norm_name,
    norm_technique_script=rb_normalization_techniques[norm_name],
    generative_model_script=rb_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=rb_train_state_reward_nex_state_df,
    dones=rb_train_dones_tensor,
    experiments_config=experiments_config,
)

Trial 13:   5%|▌         | 8000/150000 [01:14<22:02, 107.36it/s, loss=5.21e-5]
[I 2026-01-16 15:27:06,711] A new study created in RDB with name: dqn_bc_replay_buffer_min_max_data_study


Study 'dqn_bc_replay_buffer_min_max_data_study' already has 0 trials. Running 15 more...


Trial 0:  16%|█▌        | 24000/150000 [01:23<07:19, 286.49it/s, loss=4.46e-5] 
[I 2026-01-16 15:28:30,565] Trial 0 finished with value: 1.5393532521557063e-05 and parameters: {'lr': 0.00016425370029977648, 'num_hidden_neurons': 128, 'num_hidden_layers': 1}. Best is trial 0 with value: 1.5393532521557063e-05.
Trial 1:  15%|█▍        | 22000/150000 [01:57<11:24, 187.07it/s, loss=0.000134]
[I 2026-01-16 15:30:28,253] Trial 1 finished with value: 1.205380613100715e-05 and parameters: {'lr': 0.0005274213736000159, 'num_hidden_neurons': 128, 'num_hidden_layers': 5}. Best is trial 1 with value: 1.205380613100715e-05.
Trial 2:  15%|█▍        | 22000/150000 [01:45<10:14, 208.21it/s, loss=0.00035] 
[I 2026-01-16 15:32:13,993] Trial 2 finished with value: 1.4328326869872399e-05 and parameters: {'lr': 0.0001287624360615451, 'num_hidden_neurons': 128, 'num_hidden_layers': 3}. Best is trial 1 with value: 1.205380613100715e-05.
Trial 3:  18%|█▊        | 27000/150000 [02:19<10:35, 193.69it/s, loss=0.

#### Robust

In [17]:
norm_name = 'robust'

conduct_dqn_bc_experiment(
    dataset_name='replay_buffer',
    norm_technique_name=norm_name,
    norm_technique_script=rb_normalization_techniques[norm_name],
    generative_model_script=rb_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=rb_train_state_reward_nex_state_df,
    dones=rb_train_dones_tensor,
    experiments_config=experiments_config,
)

Trial 14:   4%|▍         | 6000/150000 [00:32<12:55, 185.71it/s, loss=5.51e-5]
[I 2026-01-16 15:47:16,323] A new study created in RDB with name: dqn_bc_replay_buffer_robust_data_study


Study 'dqn_bc_replay_buffer_robust_data_study' already has 0 trials. Running 15 more...


Trial 0:  38%|███▊      | 57000/150000 [04:54<07:59, 193.76it/s, loss=0.135] 
[I 2026-01-16 15:52:10,605] Trial 0 finished with value: 0.02224697545170784 and parameters: {'lr': 0.0005774595567811455, 'num_hidden_neurons': 128, 'num_hidden_layers': 3}. Best is trial 0 with value: 0.02224697545170784.
Trial 1:  61%|██████▏   | 92000/150000 [08:55<05:37, 171.89it/s, loss=0.0896]
[I 2026-01-16 16:01:05,898] Trial 1 finished with value: 0.020697856321930885 and parameters: {'lr': 0.0008728586083898291, 'num_hidden_neurons': 64, 'num_hidden_layers': 5}. Best is trial 1 with value: 0.020697856321930885.
Trial 2:  34%|███▍      | 51000/150000 [04:12<08:09, 202.16it/s, loss=0.102] 
[I 2026-01-16 16:05:18,257] Trial 2 finished with value: 0.024306025356054306 and parameters: {'lr': 0.0003798849864341881, 'num_hidden_neurons': 96, 'num_hidden_layers': 3}. Best is trial 1 with value: 0.020697856321930885.
Trial 3:  41%|████▏     | 62000/150000 [06:14<08:51, 165.45it/s, loss=0.444] 
[I 2026-01-16 

#### Standard

In [18]:
norm_name = 'standard'

conduct_dqn_bc_experiment(
    dataset_name='replay_buffer',
    norm_technique_name=norm_name,
    norm_technique_script=rb_normalization_techniques[norm_name],
    generative_model_script=rb_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=rb_train_state_reward_nex_state_df,
    dones=rb_train_dones_tensor,
    experiments_config=experiments_config,
)

[I 2026-01-16 16:28:15,584] A new study created in RDB with name: dqn_bc_replay_buffer_standard_data_study


Study 'dqn_bc_replay_buffer_standard_data_study' already has 0 trials. Running 15 more...


Trial 0:  37%|███▋      | 56000/150000 [05:21<08:58, 174.44it/s, loss=0.0256]
[I 2026-01-16 16:33:36,716] Trial 0 finished with value: 0.008889329619705677 and parameters: {'lr': 0.0006142147237680668, 'num_hidden_neurons': 32, 'num_hidden_layers': 5}. Best is trial 0 with value: 0.008889329619705677.
Trial 1:  19%|█▉        | 29000/150000 [02:30<10:28, 192.62it/s, loss=0.0333]
[I 2026-01-16 16:36:07,354] Trial 1 finished with value: 0.010815651156008244 and parameters: {'lr': 0.00017718567703612302, 'num_hidden_neurons': 32, 'num_hidden_layers': 4}. Best is trial 0 with value: 0.008889329619705677.
Trial 2:  48%|████▊     | 72000/150000 [05:54<06:23, 203.35it/s, loss=0.0476]
[I 2026-01-16 16:42:01,506] Trial 2 finished with value: 0.00997497234493494 and parameters: {'lr': 0.00012798878886221876, 'num_hidden_neurons': 64, 'num_hidden_layers': 3}. Best is trial 0 with value: 0.008889329619705677.
Trial 3:  22%|██▏       | 33000/150000 [03:28<12:17, 158.65it/s, loss=0.108] 
[I 2026-01-1

### Final Policy

In [19]:
fp_train_df = pd.read_parquet('../../../data/final_policy_episodes/fp_train_next_states_included.parquet').drop(columns=['episode'])
fp_train_state_reward_nex_state_df = fp_train_df.drop(columns=['done'])
fp_train_dones_tensor = torch.tensor(fp_train_df['done']).float()

fp_normalization_techniques = {
    'raw': None,
    'max_abs': torch.jit.load(f'../../../models/DQN/final_policy/normalization/max_abs_normalization.pt'),
    'min_max': torch.jit.load(f'../../../models/DQN/final_policy/normalization/min_max_normalization.pt'),
    'robust': torch.jit.load(f'../../../models/DQN/final_policy/normalization/robust_normalization.pt'),
    'standard': torch.jit.load(f'../../../models/DQN/final_policy/normalization/standard_normalization.pt'),
}

fp_generative_models_scripts = {
    'raw': torch.jit.load(f'../../../models/BC/final_policy/BC_raw.pt'),
    'max_abs': torch.jit.load(f'../../../models/BC/final_policy/BC_max_abs.pt'),
    'min_max': torch.jit.load(f'../../../models/BC/final_policy/BC_min_max.pt'),
    'robust': torch.jit.load(f'../../../models/BC/final_policy/BC_robust.pt'),
    'standard': torch.jit.load(f'../../../models/BC/final_policy/BC_standard.pt'),
}

Trial 14:  21%|██        | 31000/150000 [03:11<12:15, 161.70it/s, loss=0.0216]


#### Raw Data

In [20]:
norm_name = 'raw'

conduct_dqn_bc_experiment(
    dataset_name='final_policy',
    norm_technique_name=norm_name,
    norm_technique_script=fp_normalization_techniques[norm_name],
    generative_model_script=fp_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=fp_train_state_reward_nex_state_df,
    dones=fp_train_dones_tensor,
    experiments_config=experiments_config,
)

[I 2026-01-16 17:09:20,701] A new study created in RDB with name: dqn_bc_final_policy_raw_data_study


Study 'dqn_bc_final_policy_raw_data_study' already has 0 trials. Running 15 more...


Trial 0:  47%|████▋     | 70000/150000 [06:23<07:18, 182.38it/s, loss=0.208] 
[I 2026-01-16 17:15:44,599] Trial 0 finished with value: 0.03840847685933113 and parameters: {'lr': 0.00021581808506991398, 'num_hidden_neurons': 64, 'num_hidden_layers': 4}. Best is trial 0 with value: 0.03840847685933113.
Trial 1:  51%|█████     | 76000/150000 [05:20<05:11, 237.38it/s, loss=1.61]  
[I 2026-01-16 17:21:04,848] Trial 1 finished with value: 0.02972286194562912 and parameters: {'lr': 0.00063341677194014, 'num_hidden_neurons': 128, 'num_hidden_layers': 2}. Best is trial 1 with value: 0.02972286194562912.
Trial 2:  41%|████▏     | 62000/150000 [05:47<08:13, 178.27it/s, loss=0.221] 
[I 2026-01-16 17:26:52,736] Trial 2 finished with value: 0.03549771010875702 and parameters: {'lr': 0.0003938613830841012, 'num_hidden_neurons': 64, 'num_hidden_layers': 4}. Best is trial 1 with value: 0.02972286194562912.
Trial 3:  53%|█████▎    | 80000/150000 [06:09<05:23, 216.35it/s, loss=0.196] 
[I 2026-01-16 17:33

#### Max-Abs

In [21]:
norm_name = 'max_abs'

conduct_dqn_bc_experiment(
    dataset_name='final_policy',
    norm_technique_name=norm_name,
    norm_technique_script=fp_normalization_techniques[norm_name],
    generative_model_script=fp_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=fp_train_state_reward_nex_state_df,
    dones=fp_train_dones_tensor,
    experiments_config=experiments_config,
)

Trial 13:   4%|▍         | 6000/150000 [00:52<21:11, 113.29it/s, loss=0.341]
[I 2026-01-16 17:43:51,711] A new study created in RDB with name: dqn_bc_final_policy_max_abs_data_study


Study 'dqn_bc_final_policy_max_abs_data_study' already has 0 trials. Running 15 more...


Trial 0:  23%|██▎       | 35000/150000 [04:04<13:22, 143.23it/s, loss=0.0018]  
[I 2026-01-16 17:47:56,164] Trial 0 finished with value: 1.5744029951747507e-05 and parameters: {'lr': 0.0001856540806724146, 'num_hidden_neurons': 96, 'num_hidden_layers': 6}. Best is trial 0 with value: 1.5744029951747507e-05.
Trial 1:  23%|██▎       | 35000/150000 [03:49<12:34, 152.32it/s, loss=5.35e-5] 
[I 2026-01-16 17:51:46,021] Trial 1 finished with value: 1.4128673683444504e-05 and parameters: {'lr': 0.0008673646439723968, 'num_hidden_neurons': 128, 'num_hidden_layers': 5}. Best is trial 1 with value: 1.4128673683444504e-05.
Trial 2:  26%|██▌       | 39000/150000 [04:36<13:06, 141.22it/s, loss=8.88e-5] 
[I 2026-01-16 17:56:22,271] Trial 2 finished with value: 1.2590386177180335e-05 and parameters: {'lr': 0.0003956750293996282, 'num_hidden_neurons': 128, 'num_hidden_layers': 6}. Best is trial 2 with value: 1.2590386177180335e-05.
Trial 3:  27%|██▋       | 40000/150000 [03:04<08:28, 216.42it/s, loss=8

#### Min-Max

In [22]:
norm_name = 'min_max'

conduct_dqn_bc_experiment(
    dataset_name='final_policy',
    norm_technique_name=norm_name,
    norm_technique_script=fp_normalization_techniques[norm_name],
    generative_model_script=fp_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=fp_train_state_reward_nex_state_df,
    dones=fp_train_dones_tensor,
    experiments_config=experiments_config,
)

Trial 14:   5%|▍         | 7000/150000 [00:41<14:05, 169.16it/s, loss=0.000373]
[I 2026-01-16 18:09:58,782] A new study created in RDB with name: dqn_bc_final_policy_min_max_data_study


Study 'dqn_bc_final_policy_min_max_data_study' already has 0 trials. Running 15 more...


Trial 0:  20%|██        | 30000/150000 [02:34<10:18, 194.04it/s, loss=0.000139]
[I 2026-01-16 18:12:33,477] Trial 0 finished with value: 1.1723608622560278e-05 and parameters: {'lr': 0.00019737210070608303, 'num_hidden_neurons': 96, 'num_hidden_layers': 3}. Best is trial 0 with value: 1.1723608622560278e-05.
Trial 1:  31%|███       | 46000/150000 [04:57<11:13, 154.43it/s, loss=7.31e-5] 
[I 2026-01-16 18:17:31,431] Trial 1 finished with value: 1.2378808605717495e-05 and parameters: {'lr': 0.0009025741640526137, 'num_hidden_neurons': 96, 'num_hidden_layers': 5}. Best is trial 0 with value: 1.1723608622560278e-05.
Trial 2:  17%|█▋        | 25000/150000 [02:38<13:14, 157.26it/s, loss=8.5e-5]  
[I 2026-01-16 18:20:10,483] Trial 2 finished with value: 1.2413157492119353e-05 and parameters: {'lr': 0.0001917054181764928, 'num_hidden_neurons': 128, 'num_hidden_layers': 4}. Best is trial 0 with value: 1.1723608622560278e-05.
Trial 3:  23%|██▎       | 35000/150000 [03:39<12:02, 159.09it/s, loss=2

#### Robust

In [23]:
norm_name = 'robust'

conduct_dqn_bc_experiment(
    dataset_name='final_policy',
    norm_technique_name=norm_name,
    norm_technique_script=fp_normalization_techniques[norm_name],
    generative_model_script=fp_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=fp_train_state_reward_nex_state_df,
    dones=fp_train_dones_tensor,
    experiments_config=experiments_config,
)

[I 2026-01-16 18:56:45,496] A new study created in RDB with name: dqn_bc_final_policy_robust_data_study


Study 'dqn_bc_final_policy_robust_data_study' already has 0 trials. Running 15 more...


Trial 0:  55%|█████▍    | 82000/150000 [06:38<05:30, 205.76it/s, loss=0.0657]
[I 2026-01-16 19:03:24,119] Trial 0 finished with value: 0.0082400506362319 and parameters: {'lr': 0.00014725141949147887, 'num_hidden_neurons': 64, 'num_hidden_layers': 2}. Best is trial 0 with value: 0.0082400506362319.
Trial 1:  37%|███▋      | 55000/150000 [06:22<11:01, 143.67it/s, loss=0.0416]
[I 2026-01-16 19:09:47,026] Trial 1 finished with value: 0.01091062743216753 and parameters: {'lr': 0.000147382902566539, 'num_hidden_neurons': 128, 'num_hidden_layers': 5}. Best is trial 0 with value: 0.0082400506362319.
Trial 2:  45%|████▍     | 67000/150000 [05:33<06:53, 200.96it/s, loss=0.0345]
[I 2026-01-16 19:15:20,512] Trial 2 finished with value: 0.00875849835574627 and parameters: {'lr': 0.0001540795741930424, 'num_hidden_neurons': 64, 'num_hidden_layers': 2}. Best is trial 0 with value: 0.0082400506362319.
Trial 3:  29%|██▊       | 43000/150000 [03:02<07:34, 235.21it/s, loss=0.181] 
[I 2026-01-16 19:18:23

#### Standard

In [24]:
norm_name = 'standard'

conduct_dqn_bc_experiment(
    dataset_name='final_policy',
    norm_technique_name=norm_name,
    norm_technique_script=fp_normalization_techniques[norm_name],
    generative_model_script=fp_generative_models_scripts[norm_name],
    selected_features=selected_features,
    output_model_name=f'{dqn_model_variation}_{norm_name.lower()}',
    train_df=fp_train_state_reward_nex_state_df,
    dones=fp_train_dones_tensor,
    experiments_config=experiments_config,
)

Trial 14:   7%|▋         | 10000/150000 [00:37<08:46, 265.82it/s, loss=0.105]
[I 2026-01-16 19:33:17,152] A new study created in RDB with name: dqn_bc_final_policy_standard_data_study


Study 'dqn_bc_final_policy_standard_data_study' already has 0 trials. Running 15 more...


Trial 0:  35%|███▌      | 53000/150000 [04:40<08:32, 189.09it/s, loss=0.0134] 
[I 2026-01-16 19:37:57,544] Trial 0 finished with value: 0.0037968840915709734 and parameters: {'lr': 0.00013533005350369772, 'num_hidden_neurons': 128, 'num_hidden_layers': 2}. Best is trial 0 with value: 0.0037968840915709734.
Trial 1:  41%|████▏     | 62000/150000 [05:40<08:03, 182.15it/s, loss=0.0154] 
[I 2026-01-16 19:43:38,014] Trial 1 finished with value: 0.004441122990101576 and parameters: {'lr': 0.00011047329477516127, 'num_hidden_neurons': 64, 'num_hidden_layers': 2}. Best is trial 0 with value: 0.0037968840915709734.
Trial 2:  45%|████▌     | 68000/150000 [07:26<08:58, 152.34it/s, loss=0.00999]
[I 2026-01-16 19:51:04,468] Trial 2 finished with value: 0.002501850947737694 and parameters: {'lr': 0.0005294177108892096, 'num_hidden_neurons': 96, 'num_hidden_layers': 4}. Best is trial 2 with value: 0.002501850947737694.
Trial 3:  52%|█████▏    | 78000/150000 [08:03<07:26, 161.38it/s, loss=0.0153] 
[I 